# 71 — Train multi-modal cross-encoder reranker (Stage B)

Cascade: Stage A multi-modal bi-encoder (top-100) -> this cross-encoder -> top-20.
Reuses Stage A's `TripleJsonlDataset` + modality artifacts. v1 is single-positive
(teacher multi-positive dormant; see the builder's v1 NOTE).

Run order: 1 CONFIG -> 2 setup -> 3 build triples -> 4 train (open 5 TensorBoard in parallel) -> 6 dev-eval gate.
GATE before training: Stage A dev nDCG@20 >= 0.16 (run nb 70 Phase 5 first).

In [ ]:
# 1) CONFIG — Stage B multi-modal cross-encoder. Edit, then run cells 2-6.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

BRANCH               = 'stage-b-cross-encoder'
HUB_USER             = 'OrRim123'

# --- Stage A (trained multi-modal bi-encoder) — supplies the top-100 pool ---
# MUST match what nb 70 pushed (its RUN_NAME + '-merged') and its catalog-embed label.
STAGE_A_HUB_REPO     = 'OrRim123/recsys2026-bge-base-en-music-v1-mm-merged'
EMBED_LABEL          = 'bge-base-en-music-v1-mm-merged'

# --- Stage B reranker ---
RERANKER_BASE        = 'BAAI/bge-reranker-v2-m3'
RERANKER_HUB_REPO    = HUB_USER + '/recsys2026-mm-reranker-v1'

# --- Shared Phase 0 artifacts (MUST be the same dir nb 70 used) ---
MULTIMODAL_ARTIFACTS = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/multimodal'
TEACHER_SCORES_PATH  = MULTIMODAL_ARTIFACTS + '/teacher_scores.parquet'  # v1: multi-pos dormant
MULTIPOSITIVE_THRESHOLD = 0.85

# --- Data + training ---
TRIPLES_OUT          = 'experiments/cache/retrieval_v2/triples_reranker_mm.jsonl'
POOL_SIZE            = 100
N_NEGATIVES          = 7
EPOCHS               = 2       # production run. 2-3 is the sweet spot for adapting a pretrained
                               # reranker with single-positive labels; watch val/ndcg in cell 5 —
                               # if it peaks at epoch 1 and dips, drop to 1; it should peak at 2.
LR                   = 2e-5
LOSS                 = 'softmax'  # 'softmax' = in-group listwise CE (LCE): optimizes ranking directly,
                                  # loss descends from ln(1+N_NEGATIVES)=ln(8)~2.08 toward 0 (a real curve).
                                  # 'bce' = per-pair calibration (v1; floors at 2*ln2~1.386 under noisy labels).
LOSS_TEMPERATURE     = 1.0        # softmax sharpness: <1 sharper, >1 softer (only used when LOSS='softmax').
MAX_GRAD_NORM        = 25.0       # clip global grad norm to this (0 = off). ~25 neutralizes rare softmax
                                  # spikes (we saw grad_norm jump to 230) without throttling the productive
                                  # ~20 gradients. Logged grad_norm stays PRE-clip so spikes remain visible.
BATCH_SIZE           = 16      # Blackwell 95GB. Each step forwards bs*(1+N_NEGATIVES)=bs*8 cross-encoder
                               # pairs @ max_length, so VRAM scales with bs: 24 OOMs (>95GB),
                               # 16 ~= 67GB (safe), 12 ~= 53GB. max_length caps shapes so any OOM hits
                               # step 1 (~1 min) -> drop to 12 if 16 OOMs, or set GRADIENT_CHECKPOINTING
                               # =True to keep a larger bs. NB: for the CE, bs does NOT change the
                               # negatives seen (each row owns its 7 negs), so shrinking is loss-neutral.
MAX_LENGTH           = 512
MAX_ROWS             = 0       # 0 = all; set e.g. 200 for a quick smoke run
TRAIN_OUTPUT_DIR     = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/training/mm_reranker_v1'  # on Drive so checkpoints survive disconnects
CHECKPOINT_EVERY_N_STEPS = 1000   # rolling checkpoint to {TRAIN_OUTPUT_DIR}/checkpoint_latest every N steps (disconnect-safe)
RESUME_FROM          = ''         # set to a checkpoint dir to resume, e.g. TRAIN_OUTPUT_DIR + '/checkpoint_latest'
GRADIENT_CHECKPOINTING = False    # True -> lower VRAM (fit larger BATCH_SIZE), ~30% slower/step

# --- Validation + live metrics (mirrors Stage A; watch the curves in cell 5, TensorBoard) ---
VAL_FRACTION         = 0.05       # hold out this fraction of triples for val (0 = train on all, no val pass)
SPLIT_KEY            = 'user_id'  # leak-safe split: a user's sessions stay on one side ('session_id'|'row' also valid)
VAL_EVERY_N_STEPS    = 500        # run a val pass (loss + top1/nDCG) every N opt-steps
VAL_MAX_ROWS         = 1000       # cap rows scored per val pass so each stays ~1 min (0 = full held-out val set)
LOG_EVERY            = 25         # log train loss + in-group top1/nDCG every N opt-steps
TENSORBOARD_PORT     = 6006

# --- Catalog / retriever lookups (MUST match nb 70) ---
ITEM_DB              = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS_TYPES         = 'track_name,artist_name,album_name'
CACHE_DIR            = 'experiments/cache/retrieval_v2'
print('CONFIG set. Stage A:', STAGE_A_HUB_REPO, '-> reranker:', RERANKER_HUB_REPO, '| loss:', LOSS)

In [ ]:
# 2) Setup — clone branch + HF auth + Drive mount + cache symlink + deps.
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the Drive cache so triples + catalog embeddings persist across sessions.
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
_src = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache'
_dst = LOCAL_BASE + '/retrieval_v2'
os.makedirs(_src, exist_ok=True)
if os.path.islink(_dst): os.unlink(_dst)
elif os.path.exists(_dst):
    import shutil; shutil.rmtree(_dst)
os.symlink(_src, _dst)

# Retrieval-stack deps: build_cross_encoder_training_data imports from
# mcrs.retrieval_modules, whose __init__ eagerly loads bm25 (-> bm25s) and
# dense_local (-> sentence-transformers); without these, cell 3 ImportErrors.
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' 'bm25s'

In [ ]:
# 3) Build Stage B training triples — Stage A top-100 per training query.
# Single-positive v1 (teacher multi-pos dormant; see builder v1 NOTE). Set
# MAX_ROWS>0 in CONFIG for a quick smoke run first.
_maxrows = ('--max-rows ' + str(MAX_ROWS)) if MAX_ROWS else ''
!cd /content/recsys2026 && python -u scripts/build_cross_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --stage-a-hub-repo {STAGE_A_HUB_REPO} \
    --multimodal-artifacts {MULTIMODAL_ARTIFACTS} \
    --multipositive-threshold {MULTIPOSITIVE_THRESHOLD} \
    --output {TRIPLES_OUT} --pool-size {POOL_SIZE} \
    --embed-label {EMBED_LABEL} --cache-dir {CACHE_DIR} \
    --history-corpus-types {CORPUS_TYPES} {_maxrows} \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/ce_build_log.txt
!wc -l {TRIPLES_OUT}

In [ ]:
# 4) Train the multi-modal cross-encoder (full FT, bf16). Pushes final model -> Hub.
# Open cell 5 (TensorBoard) in PARALLEL to watch train/val loss + nDCG live.
# Checkpointing: writes {TRAIN_OUTPUT_DIR}/checkpoint_latest every CHECKPOINT_EVERY_N_STEPS
# steps + checkpoint_epoch_N per epoch (Drive-resident). On a disconnect, resume by
# setting RESUME_FROM in cell 1 to that dir and re-running this cell.
_resume   = ('--resume-from ' + RESUME_FROM) if RESUME_FROM else ''
_gradckpt = '--gradient-checkpointing' if GRADIENT_CHECKPOINTING else ''
!cd /content/recsys2026 && python -u scripts/train_cross_encoder.py \
    --triples {TRIPLES_OUT} \
    --multimodal-artifacts {MULTIMODAL_ARTIFACTS} \
    --base-model {RERANKER_BASE} \
    --output-dir {TRAIN_OUTPUT_DIR} \
    --hub-repo {RERANKER_HUB_REPO} \
    --epochs {EPOCHS} --lr {LR} --batch-size {BATCH_SIZE} \
    --loss {LOSS} --loss-temperature {LOSS_TEMPERATURE} --max-grad-norm {MAX_GRAD_NORM} \
    --n-negatives {N_NEGATIVES} --max-length {MAX_LENGTH} \
    --val-fraction {VAL_FRACTION} --split-key {SPLIT_KEY} \
    --val-every-n-steps {VAL_EVERY_N_STEPS} --val-max-rows {VAL_MAX_ROWS} \
    --log-every {LOG_EVERY} \
    --checkpoint-every-n-steps {CHECKPOINT_EVERY_N_STEPS} {_resume} {_gradckpt} \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/ce_train_log.txt

In [ ]:
# 5) TensorBoard — open in PARALLEL with cell 4 (training). Auto-refreshes ~30s.
# Curves to watch (same channel as nb 70's Stage A launcher):
#   train/loss   vs val/loss        — descend then plateau; a growing gap = overfit onset
#   train/ndcg_ingroup vs val/ndcg  — in-group ranking quality over the row's 8 candidates
#   train/grad_norm                 — should stay bounded; spikes = instability
# NOTE: in-group nDCG is a TRAINING-HEALTH signal over 8 candidates (1 pos + 7 negs),
# so even a random model sits around ~0.4 — it is NOT the honest dev nDCG@20. The
# real cascade gate is cell 6 (rerank top-100 -> nDCG@20 over the full catalog).
%load_ext tensorboard
%tensorboard --logdir {TRAIN_OUTPUT_DIR}/runs --port={TENSORBOARD_PORT}

In [ ]:
# 6) Dev eval — PRODUCTION query path + full metric suite (Stage A vs Stage A+B).
# FIX (2026-05-24): build dev queries via the production path — MusicCatalogDB +
# build_retrieval_query with chat-history music turns EXPANDED via id_to_metadata
# (mirrors nb 70 cell 7 and the deployed wRRF path). The prior version fed raw
# track-id UUIDs into [HISTORY] for ~87% of dev turns (out-of-distribution), which
# alone collapsed Stage A to 0.078. This is eval-only — no retrain, runs on the
# model already on the Hub.
import sys, math, json, time
import numpy as np
from datasets import load_dataset
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.crs_baseline import build_retrieval_query
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules.dense_multimodal_local import DENSE_MULTIMODAL_LOCAL
from mcrs.rerankers.multimodal_cross_encoder_rerank import MULTIMODAL_RERANKER

CORPUS      = CORPUS_TYPES.split(',')
N_EVAL      = 500     # dev turns to score
POOL        = 500     # retrieve this deep (enables recall@200/300/500 + bigger-pool analysis)
RERANK_POOL = 100     # reranker reorders the top-RERANK_POOL into top-20 (matches config 182)

# --- production-path query construction (identical to nb 70 cell 7) ---
item_db = MusicCatalogDB(dataset_name=ITEM_DB, split_types=['all_tracks'], corpus_types=CORPUS)

def _build_history_for_user_at(convs, user_pos):
    hist = []
    for prev in convs[:user_pos]:
        role, content = prev.get('role'), (prev.get('content') or '')
        if role == 'user':
            hist.append({'role': 'user', 'content': content})
        elif role == 'music':
            exp = item_db.id_to_metadata(content) if content in item_db.metadata_dict else content
            hist.append({'role': 'assistant', 'content': exp})
        elif role == 'assistant':
            hist.append({'role': 'assistant', 'content': content})
    return hist

test_sess = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
queries, golds, user_ids = [], [], []
for sess in test_sess:
    if len(queries) >= N_EVAL:
        break
    convs = sess.get('conversations', [])
    uid = sess.get('user_id')
    for up, turn in enumerate(convs):
        if len(queries) >= N_EVAL:
            break
        if turn.get('role') != 'user':
            continue
        if up + 1 >= len(convs) or convs[up + 1].get('role') != 'music':
            continue
        uq, gold = (turn.get('content') or ''), convs[up + 1].get('content')
        if not uq or not gold:
            continue
        hist = _build_history_for_user_at(convs, up)
        cg = sess.get('conversation_goal') or {}
        goal = (cg.get('listener_goal') or '').strip() or None
        queries.append(build_retrieval_query(hist + [{'role': 'user', 'content': uq}],
                                             mode='bge_m3_structured', goal_text=goal,
                                             user_profile=sess.get('user_profile')))
        golds.append(gold)
        user_ids.append(str(uid) if uid is not None else None)
print(f'[dev eval] {len(queries)} queries via production path; '
      f'{sum(1 for u in user_ids if u)} warm / {sum(1 for u in user_ids if u is None)} cold users')

# --- retrieve (dense, deep) + rerank top-RERANK_POOL ---
retriever = DENSE_MULTIMODAL_LOCAL(
    dataset_name=ITEM_DB, split_types=['all_tracks'], corpus_types=CORPUS,
    cache_dir=CACHE_DIR, model_dir=STAGE_A_HUB_REPO, embed_label=EMBED_LABEL,
    multimodal_artifacts=MULTIMODAL_ARTIFACTS,
)
reranker = MULTIMODAL_RERANKER(
    model_dir=RERANKER_HUB_REPO, multimodal_artifacts=MULTIMODAL_ARTIFACTS,
    item_db_name=ITEM_DB, track_split_types=['all_tracks'], corpus_types=CORPUS,
    cache_dir=CACHE_DIR,
)
candP = retriever.batch_text_to_item_retrieval(queries, topk=POOL, user_ids=user_ids)
cand100 = [c[:RERANK_POOL] for c in candP]
reranked20 = reranker.rerank(queries, cand100, topk=20, user_ids=user_ids)
if hasattr(retriever, 'flush_query_cache'):
    retriever.flush_query_cache()

# --- metrics ---
def _ndcg_at(ranked, gold, k):
    r = ranked[:k]
    return 1.0 / math.log2(r.index(gold) + 2) if gold in r else 0.0

def _rank(ranked, gold):
    return (ranked.index(gold) + 1) if gold in ranked else None

sa_ndcg20  = np.array([_ndcg_at(c, g, 20) for c, g in zip(candP, golds)])
sa_ndcg10  = np.array([_ndcg_at(c, g, 10) for c, g in zip(candP, golds)])
sab_ndcg20 = np.array([_ndcg_at(r, g, 20) for r, g in zip(reranked20, golds)])
sa_mrr     = np.array([(1.0 / _rank(c, g)) if _rank(c, g) else 0.0 for c, g in zip(candP, golds)])
stage_a, stage_ab = float(sa_ndcg20.mean()), float(sab_ndcg20.mean())

print('\n=== Ranking (Stage A = dense; Stage A+B = + cross-encoder rerank of top-%d) ===' % RERANK_POOL)
print(f'  Stage A   nDCG@20={stage_a:.4f}  nDCG@10={sa_ndcg10.mean():.4f}  MRR={sa_mrr.mean():.4f}')
print(f'  Stage A+B nDCG@20={stage_ab:.4f}  (lift {stage_ab - stage_a:+.4f}; gate >= +0.04)')

# recall@K over the dense pool
print('\n=== Recall (gold present in dense top-K), n=%d ===' % len(golds))
recK = {}
for k in (5, 10, 20, 50, 100, 200, 300, 500):
    recK[k] = float(np.mean([1.0 if (g in c[:k]) else 0.0 for c, g in zip(candP, golds)]))
    print(f'  recall@{k:<3} = {recK[k]:.4f}')
print(f'  CEILING: nDCG@20 <= recall@{RERANK_POOL} = {recK[RERANK_POOL]:.4f} (reranker only sees top-{RERANK_POOL}).')
print(f'  recall keeps climbing to @500={recK[500]:.4f} -> a deeper rerank pool would raise the ceiling.')

# reranker realized headroom + regression
in20     = np.array([(g in c[:20]) for c, g in zip(candP, golds)])
in21_100 = np.array([(g in c[:RERANK_POOL] and g not in c[:20]) for c, g in zip(candP, golds)])
sab_in20 = np.array([(g in r[:20]) for r, g in zip(reranked20, golds)])
headroom_n, realized = int(in21_100.sum()), int((in21_100 & sab_in20).sum())
regress_n, demoted   = int(in20.sum()), int((in20 & ~sab_in20).sum())
print('\n=== Reranker behavior ===')
print(f'  rerankable golds (dense rank 21-{RERANK_POOL}): {headroom_n} '
      f'-> pulled into top-20 by Stage B: {realized} '
      f'({(realized / headroom_n if headroom_n else 0):.1%} of opportunity)')
print(f'  golds dense had in top-20: {regress_n} '
      f'-> DEMOTED out of top-20 by Stage B: {demoted} '
      f'({(demoted / regress_n if regress_n else 0):.1%})  (regression check; want ~0%)')

# paired-bootstrap CI on the Stage B lift
def _paired_ci(a, b, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    d = b - a
    boot = d[rng.integers(0, len(d), size=(n, len(d)))].mean(axis=1)
    return float(d.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
_obs, _lo, _hi = _paired_ci(sa_ndcg20, sab_ndcg20)
print('\n=== Significance of the Stage B lift (paired bootstrap, 2000 resamples) ===')
print(f'  delta nDCG@20 = {_obs:+.4f}   95% CI [{_lo:+.4f}, {_hi:+.4f}]   '
      f'-> {"SIGNIFICANT (CI excludes 0)" if _lo > 0 else "NOT significant (CI includes 0)"}')

print('\n[dev eval] done. queries/golds/user_ids/candP + retriever are in memory for cell 7 (per-stream recall).')

In [ ]:
# 7) Per-stream recall — does the multi-modal DENSE tower earn its place vs BM25,
# and what does the DEPLOYED fused retriever (config 182) actually ceiling at?
# Reuses queries/golds/user_ids + the dense `retriever` from cell 6 — RUN CELL 6 FIRST.
assert 'queries' in globals() and 'golds' in globals() and 'retriever' in globals(), \
    'Run cell 6 first — it builds queries/golds/user_ids and the dense retriever.'
import numpy as np
from mcrs.retrieval_modules import load_retrieval_module

def _recall_at(cands, k):
    return float(np.mean([1.0 if (g in c[:k]) else 0.0 for c, g in zip(cands, golds)]))

streams = {}
# dense multi-modal tower alone (already built in cell 6)
streams['dense_mm'] = retriever.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids)

# BM25 alone (lexical; ignores user_ids)
try:
    _bm25 = load_retrieval_module('bm25', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR)
    streams['bm25'] = _bm25.batch_text_to_item_retrieval(queries, topk=100)
except Exception as e:
    print('[per-stream] bm25 failed:', type(e).__name__, e)

# fused = the DEPLOYED Stage A (config 182): wRRF(BM25 + dense_mm)
try:
    _fused = load_retrieval_module(
        'wrrf_bm25_multimodal_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
        extra_config={'mm_model_dir': STAGE_A_HUB_REPO, 'embed_label': EMBED_LABEL,
                      'multimodal_artifacts': MULTIMODAL_ARTIFACTS},
    )
    streams['fused (deployed)'] = _fused.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids)
except Exception as e:
    print('[per-stream] fused failed:', type(e).__name__, e)

print('\n=== Per-stream recall (n=%d) ===' % len(golds))
print(f'  {"stream":<18}recall@20  recall@100')
for name, cands in streams.items():
    print(f'  {name:<18}{_recall_at(cands, 20):.4f}     {_recall_at(cands, 100):.4f}')
print('\nReads:')
print('  - dense_mm < bm25  -> the multi-modal tower underperforms lexical alone (Stage A quality issue).')
print('  - fused recall@100 is the DEPLOYED ceiling (what nb 73 / config 182 feeds Stage B); if it is')
print('    well above either stream, BM25 + dense recover different golds and fusion is doing its job.')
print('  - Lift the ceiling further: deeper pool (cell-6 recall@500) and/or a richer recall ensemble.')

## Deploy the cascade (Blind-A / dev YAML)

The cascade is config, not new code (the reranker is registered in
`mcrs/rerankers`). In the inference YAML (nb 73 / run_inference_*), set:

```yaml
retrieval_type: wrrf_bm25_multimodal_v1
retrieval_topk: 100                       # feed the reranker a top-100 pool
reranker_type: multimodal_cross_encoder
reranker_model_path: OrRim123/recsys2026-mm-reranker-v1
reranker_multimodal_artifacts: /content/drive/MyDrive/recsys2026_retrieval_v2_cache/multimodal
```

`user_ids` flow to the reranker automatically via `batch_chat`.